# Statistical Validation: Alternative vs. Commercial Music

This notebook validates whether the audio feature differences between `alternative` and `comercial` groups are statistically significant and practically meaningful.

**Tests used:**
- **Mann-Whitney U** — non-parametric test for comparing two independent groups (does not assume normal distribution)
- **Cohen's d** — effect size measure to assess practical significance beyond p-values

**Features tested:** `valence`, `energy`, `acousticness`, `danceability`

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from riffscope.config import PROCESSED_DATA_DIR

2026-06-15 16:04:05.051 | INFO     | riffscope.config:<module>:11 - PROJ_ROOT path is: /home/axeletl/Documents/projects/riffScope-insights-pipeline/riffScope-insights-pipeline


In [3]:
df = pd.read_csv(PROCESSED_DATA_DIR / "dataset_clean.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1695 entries, 0 to 1694
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            1695 non-null   str    
 1   name          1695 non-null   str    
 2   artists       1695 non-null   str    
 3   acousticness  1695 non-null   float64
 4   danceability  1695 non-null   float64
 5   energy        1695 non-null   float64
 6   valence       1695 non-null   float64
 7   group         1695 non-null   str    
 8   release_year  1695 non-null   int64  
dtypes: float64(4), int64(1), str(4)
memory usage: 119.3 KB


## Split Groups

In [4]:
alternative = df[df["group"] == "alternative"]
comercial = df[df["group"] == "comercial"]

print(f"Alternative tracks: {len(alternative)}")
print(f"Commercial tracks:  {len(comercial)}")

Alternative tracks: 964
Commercial tracks:  731


## Mann-Whitney U + Cohen's d

Cohen's d interpretation:

| d | Effect size |
|---|---|
| 0.2 | small |
| 0.5 | medium |
| 0.8 | large |

In [5]:
FEATURES = ["valence", "energy", "acousticness", "danceability"]

def cohen_d(a, b):
    pooled_std = np.sqrt((a.std(ddof=1) ** 2 + b.std(ddof=1) ** 2) / 2)
    return (a.mean() - b.mean()) / pooled_std

results = []
for feature in FEATURES:
    a = alternative[feature]
    b = comercial[feature]
    u_stat, p_value = stats.mannwhitneyu(a, b, alternative="two-sided")
    d = cohen_d(a, b)
    results.append({
        "feature": feature,
        "mean_alternative": round(a.mean(), 4),
        "mean_comercial": round(b.mean(), 4),
        "U_statistic": round(u_stat, 2),
        "p_value": round(p_value, 6),
        "cohen_d": round(d, 4),
        "significant": p_value < 0.05
    })

results_df = pd.DataFrame(results)
results_df

,feature,mean_alternative,mean_comercial,U_statistic,p_value,cohen_d,significant
0,valence,0.4801,0.5670,280556.5,0.0,-0.3565,True
1,energy,0.7338,0.6668,439079.5,0.0,0.3323,True
2,acousticness,0.1536,0.2531,229899.0,0.0,-0.3921,True
3,danceability,0.4866,0.6747,121248.0,0.0,-1.3008,True


## Interpretation

- **p_value < 0.05** → the difference between groups is statistically significant
- **cohen_d** → how large the difference is in practical terms
- A positive `cohen_d` means `alternative` scores higher on that feature; negative means `comercial` scores higher

---

### What does `p_value = 0.0` mean?

All 4 features return p ≈ 0 (too small to display at 6 decimal places). This means the differences between `alternative` and `comercial` are **not due to chance** — they are statistically real across the 1,695 tracks in the dataset.

---

### What does `cohen_d` tell us?

The **sign** indicates which group scores higher. The **magnitude** indicates how large the difference is.

| Feature | cohen_d | Reading |
|---|---|---|
| `danceability` | -1.30 | **Large** — commercial music is significantly more danceable. Strongest finding in the analysis. |
| `acousticness` | -0.39 | Small-medium — commercial scores higher (reggaeton, latin-pop carry more acoustic elements than metal or hard-rock) |
| `valence` | -0.36 | Small-medium — commercial music is emotionally more positive |
| `energy` | +0.33 | Small-medium — alternative music is more intense and energetic |

---

### What story do these numbers tell?

Alternative music is more intense, emotionally darker, and less danceable. Commercial music is more positive, more danceable, and surprisingly more acoustic. `danceability` with d = -1.30 is the strongest differentiator — expected given that the commercial group includes reggaeton, trap, k-pop, and dance genres.

These results support the research question: alternative music carries a clearly distinct sonic signature, not just a cultural perception.